# 1일차 5교시 · 과적합과 데이터 누수

1. 결정 트리 깊이를 바꿔 가며 **과적합**을 관찰합니다.
2. 검증 곡선과 학습 곡선을 그립니다.
3. **데이터 누수**가 성능을 얼마나 부풀리는지 직접 확인합니다.

그래프의 글자는 Colab에서 한글 폰트가 깨지지 않도록 영어로 적었습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, validation_curve, learning_curve, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

## 1. 깊이에 따른 학습·테스트 정확도

In [ ]:
rows = []
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    train_acc, test_acc = tree.score(X_train, y_train), tree.score(X_test, y_test)
    rows.append({"최대 깊이": "제한 없음" if depth is None else depth,
                 "학습 정확도": round(train_acc, 3),
                 "테스트 정확도": round(test_acc, 3),
                 "차이": round(train_acc - test_acc, 3)})
pd.DataFrame(rows)

- 깊이를 늘리면 학습 정확도는 1.0까지 오르지만 테스트 정확도는 오히려 떨어집니다 → **과적합**
- 설명을 위해 테스트 데이터로 비교했지만, 실제로는 **검증 데이터나 교차 검증**으로 깊이를 고릅니다(아래 2번).

## 2. 검증 곡선: 교차 검증으로 깊이 고르기

학습 데이터 안에서 5겹 교차 검증으로 깊이별 성능을 봅니다. 테스트 데이터는 쓰지 않습니다.

In [ ]:
depths = np.arange(1, 13)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=0), X_train, y_train,
    param_name="max_depth", param_range=depths, cv=5)

plt.figure(figsize=(6, 4))
plt.plot(depths, train_scores.mean(axis=1), marker="o", label="train")
plt.plot(depths, val_scores.mean(axis=1), marker="o", label="validation (5-fold CV)")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Validation curve")
plt.legend()
plt.show()
best = depths[val_scores.mean(axis=1).argmax()]
print("검증 성능이 가장 좋은 깊이:", best)

## 3. 학습 곡선: 데이터가 늘면 나아질까

In [ ]:
sizes, tr, va = learning_curve(
    DecisionTreeClassifier(max_depth=None, random_state=0), X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 6), cv=5)

plt.figure(figsize=(6, 4))
plt.plot(sizes, tr.mean(axis=1), marker="o", label="train")
plt.plot(sizes, va.mean(axis=1), marker="o", label="validation (5-fold CV)")
plt.xlabel("training samples")
plt.ylabel("accuracy")
plt.title("Learning curve (unlimited depth tree)")
plt.legend()
plt.show()

학습 정확도는 계속 1.0이고 검증 정확도와 간격이 있습니다 → 깊이 제한 없는 트리는 과적합 상태입니다. 데이터가 늘면 간격이 줄어드는지도 확인해 보세요.

## 4. 데이터 누수: 무작위 데이터로 99% 정확도 만들기

**특성도 라벨도 완전히 무작위**인 데이터를 만듭니다. 제대로 평가하면 정확도는 **0.5 근처**여야 합니다.

그런데 **분할 전에 전체 데이터로 특성을 고르면**(라벨 정보를 사용하는 전처리) 성능이 크게 부풀려집니다.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

rng = np.random.RandomState(0)
X_noise = rng.normal(size=(100, 10000))     # 100명, 무의미한 특성 10,000개
y_noise = rng.randint(0, 2, size=100)       # 무작위 라벨

# 나쁜 예: 전체 데이터(라벨 포함)로 특성 20개를 먼저 고른 뒤 교차 검증
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=5).mean()

# 좋은 예: 특성 선택을 Pipeline 안에 넣어, 교차 검증의 학습 부분에서만 수행
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
correct = cross_val_score(pipe, X_noise, y_noise, cv=5).mean()

print(f"누수 있는 평가 정확도: {leaky:.2f}")
print(f"올바른 평가 정확도  : {correct:.2f}")

- 누수가 있으면 **의미 없는 데이터로도 높은 정확도**가 나옵니다. 운영에 가면 당연히 무너집니다.
- 해결책: **분할 먼저**, 라벨이나 분포를 쓰는 전처리(특성 선택, 스케일링, 결측 대체)는 **Pipeline 안에서** 학습 데이터로만 fit합니다.

## 5. [선택] 연습

`DecisionTreeClassifier`의 `min_samples_leaf` 값을 1, 5, 20, 50으로 바꿔 가며 학습·테스트 정확도 차이가 어떻게 변하는지 확인해 보세요.

In [ ]:
# 여기에 코드를 작성하세요